# ❤️ Heart Disease Prediction — Model Training

This notebook mirrors `src/train.py` and produces the artifacts the app consumes.

In [ ]:
import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

from src.config import (
    ENSEMBLE_PIPELINE, EXPLAINER_PATH, FEATURE_IMPORTANCES_JSON,
    INDIVIDUAL_MODELS, METRICS_JSON, RANDOM_STATE, TEST_SIZE,
)
from src.preprocessing import build_preprocessor, clean_data, load_data, prepare_features_target
from src.evaluation import evaluate_classifier, roc_points, save_roc_curves, save_confusion_matrices, write_metrics

## 1. Load & clean data

In [ ]:
df = clean_data(load_data())
print('Shape:', df.shape)
print('Target distribution:')
print(df['target'].value_counts())

## 2. Exploratory data analysis

In [ ]:
df.describe().T

# missing values
print('Missing values:')
print(df.isna().sum().sum())

## 3. Preprocessing & stratified split

In [ ]:
X, y = prepare_features_target(df)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')

## 4. Train individual models

In [ ]:
preprocessor = build_preprocessor()

models = {
    'Logistic Regression': (LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE), 'lr'),
    'Random Forest': (RandomForestClassifier(n_estimators=300, min_samples_split=5, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1), 'rf'),
    'XGBoost': (XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, subsample=0.9, colsample_bytree=0.8, eval_metric='logloss', random_state=RANDOM_STATE), 'xgb'),
}

from sklearn.pipeline import Pipeline
fitted = {}
for name, (est, key) in models.items():
    pipe = Pipeline(steps=[('prep', preprocessor), (key, est)])
    pipe.fit(X_train, y_train)
    fitted[name] = pipe
    print('Fitted', name)

## 5. Soft-voting ensemble

In [ ]:
ensemble = VotingClassifier(
    estimators=[(k.lower().replace(' ', '_'), v) for k, v in fitted.items()],
    voting='soft', n_jobs=-1,
)
ensemble.fit(X_train, y_train)
print('Fitted ensemble')

## 6. Evaluate on the held-out test set

In [ ]:
metrics, roc_data, cm_data = {}, {}, {}
all_models = {**fitted, 'Ensemble (Soft Voting)': ensemble}
for name, model in all_models.items():
    metrics[name] = evaluate_classifier(model, X_test, y_test)
    pts = roc_points(model, X_test, y_test)
    pts['roc_auc'] = metrics[name]['roc_auc']
    roc_data[name] = pts
    cm_data[name] = metrics[name]['confusion_matrix']
    m = metrics[name]
    print(f"{name:26s} acc={m['accuracy']:.3f} f1={m['f1']:.3f} auc={m['roc_auc']:.3f}")

## 7. Save artifacts

In [ ]:
write_metrics(metrics, roc_data)
save_roc_curves(roc_data)
save_confusion_matrices(cm_data)

joblib.dump(ensemble, ENSEMBLE_PIPELINE)
joblib.dump(fitted, INDIVIDUAL_MODELS)
print('Saved ensemble + individual models and metric artifacts.')

## 8. SHAP explainability

In [ ]:
import shap

X_trans = preprocessor.named_steps['prep'].fit_transform(X_train)
feature_names = [str(f) for f in preprocessor.named_steps['prep'].get_feature_names_out()]

xgb_explainer = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, subsample=0.9, colsample_bytree=0.8, eval_metric='logloss', random_state=RANDOM_STATE)
xgb_explainer.fit(X_trans, y_train)
explainer = shap.TreeExplainer(xgb_explainer)

joblib.dump(
    {'explainer': explainer, 'preprocessor': preprocessor.named_steps['prep'], 'feature_names': feature_names, 'model': xgb_explainer},
    EXPLAINER_PATH,
)
print('Explainable saved at', EXPLAINER_PATH)